# Decision trees, random forests, and boosting

Compare one bounded-depth tree, an ensemble of randomized trees, and sequential boosting. Wine is bundled with scikit-learn. This is a small example of model comparison, not an application benchmark.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.tree import DecisionTreeClassifier

SEED = 42
data = load_wine()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target,
    test_size=0.25, stratify=data.target, random_state=SEED)
models = {
    'decision tree': DecisionTreeClassifier(max_depth=4, random_state=SEED),
    'random forest': RandomForestClassifier(n_estimators=80, max_depth=5, random_state=SEED, n_jobs=1),
    'gradient boosting': GradientBoostingClassifier(n_estimators=60, max_depth=2, random_state=SEED),
}
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)


## Compare identical folds

The tree depth and ensemble sizes are fixed teaching settings. Trees split on feature thresholds and do not require standardization in this example.


In [ ]:
rows = []
for name, model in models.items():
    result = cross_validate(model, X_train, y_train, cv=cv,
        scoring='accuracy', return_train_score=True, n_jobs=1)
    rows.append({'model': name, 'train_accuracy': result['train_score'].mean(),
                 'cv_accuracy': result['test_score'].mean(),
                 'cv_std': result['test_score'].std()})
comparison = pd.DataFrame(rows).set_index('model')
display(comparison.round(3))
selected_name = comparison['cv_accuracy'].idxmax()
selected = models[selected_name].fit(X_train, y_train)
prediction = selected.predict(X_test)
print('Selected:', selected_name, 'Test accuracy:', round(accuracy_score(y_test, prediction), 3))
print(classification_report(y_test, prediction, zero_division=0))


## Inspect the selected model after final evaluation

Permutation importance measures a drop in score when a column is shuffled. Correlated features can share information, so this is not a causal explanation or a rule for selecting features from the test set.


In [ ]:
importance = permutation_importance(selected, X_test, y_test,
    scoring='accuracy', n_repeats=5, random_state=SEED, n_jobs=1)
order = importance.importances_mean.argsort()[-6:]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh([data.feature_names[i] for i in order], importance.importances_mean[order],
        xerr=importance.importances_std[order])
ax.set_xlabel('Held-out accuracy drop after shuffling')
fig.tight_layout()
plt.show()


## Practice

Use a training-only grid search for depth or minimum leaf size. Explain the difference between bagging-style random forests and gradient boosting. Treat the final test result as an estimate with substantial uncertainty on this small sample.

**References:** [trees](https://scikit-learn.org/stable/modules/tree.html), [ensembles](https://scikit-learn.org/stable/modules/ensemble.html).
